In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

In [ ]:
import torch
import torchvision
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import time
from pathlib import Path

# The standard device check — you'll use this pattern in every PyTorch notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version:     {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")

# Tensor Q1:

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

b = torch.zeros(2, 3)
c = torch.ones(4)
print(a.shape, a.dtype, a.device)
print(b.shape, b.dtype, b.device)
print(c.shape, c.dtype, c.device)

the tensors above are in cpu right now. If we were running a loop, the device and weights better be in the GPU, because it requires faster computation. 

# Tensor Q2:

In [ ]:
x = torch.tensor([1.0, 4.0, 9.0, 16.0, 25.0])

# square root

sq_root = torch.sqrt(x)
print(sq_root)
print(x.sum())
print(x.mean())
print(x.argmax())


In [ ]:
Argmax() gives the position where the value is located. 

# Tensor Q3:

In [ ]:
a_gpu   = a.to(device)
print(f"a_gpu device: {a_gpu.device}")

a_back  = a_gpu.cpu()
print(a_back.device)

a_numpy = a_back.numpy()
print(f"numpy type: {type(a_numpy)}")
print(f"numpy values:\n{a_numpy}")

Because, NumPy needs cpu. NumPy was made for CPU only (regular computer memory), on the other hand, PyTorch can use GPU memory or CPU memory

# Tensor Q4:

In [ ]:
t = torch.arange(24).float()

# reshaping t to (4,6)
a_reshape = t.reshape(4,6)
print(a_reshape.shape)

# reshaping t to (2,3,4)
b_reshape = t.reshape((2, 3, 4))
print(b_reshape.shape)

# Take your result from step 1 and add a new dimension at position 0. Print the new shape.
new_dimension = a_reshape.unsqueeze(0)
print(new_dimension.shape)


A single image tensor typically has shape (channels, height, width). Neural networks expect batches with shape (batch_size, channels, height, width). It is because we need to add one more dimension to process one image in a single shape as neural network expects a batch. So, when we add a new dimension, it will imitate a batch.

# Tensor #5

In [ ]:
np_a = np.array([[1.0, 2.0], [3.0, 4.0]])
np_b = np.array([[5.0, 6.0], [7.0, 8.0]])

t_a  = torch.tensor(np_a, dtype=torch.float32)
t_b  = torch.tensor(np_b, dtype=torch.float32)

np_multipl = np.matmul(np_a, np_b)
pt_multipl = torch.matmul(t_a, t_b)
print(np_multipl)
print(pt_multipl)

Above it is clear that the tensor's value is in brackets, meaning its difference from numpy. Matrix multiplication helps the model transform input data into more informative features, enabling better final predictions.

# Pretrained Models: Model Q1:

In [ ]:
weights = ResNet18_Weights.DEFAULT
model   = models.resnet18(weights=weights)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

ResNet18 has roughly 11 million parameters. Training it from scratch required approximately 1.2 million labeled ImageNet images and days of multi-GPU compute. What does that tell you about the practical value of starting from pretrained weights when you're on a deadline or a budget? When we are on a deadline or a budget, starting with a pretrained weights or model helps greatly because we don't have spend that many days of compute in training and also hge size of data. Pretrained weights save us time, resources and sizable data. 

In [ ]:
print(model)

The final layer in ResNet18 is named "fc".
It is a Linear layer with 512 input features and 1000 output features.
The output size is 1000, which means the model can predict 1000 ImageNet categories.

The blocks named layer1, layer2, layer3, and layer4 are the deep feature extractor part of ResNet18. They are made of BasicBlock modules containing convolution, batch normalization, and ReLU layers.

A network is called "deep" because it has many layers stacked together.
Earlier layers learn simple features like edges and colors.
Middle layers learn shapes and textures.
Deeper layers learn more complex patterns like object parts and whole-object features.

# Model Q3:

In [ ]:
# This code snippet prepares a trained neural network for inference

# this line of code sets the hardware device to the one we defined initially. It is usually between GPU and CPU, because we initially chose GPU, the model will run on GPU. 
model = model.to(device)

# This line sets the model to evaluation mode. It is an important step in selecting the inference, as its layers behave differently during training vs. testing. 
#There are two main layers that behave differently: 
# a) Dropout - randomly sets zeros to some parts of the input data in training, and in testing, it disables this random dropping and acts as a "passthrough" activing all parts of neurons.
# b) Batch normalization layer uses current batch statistics in training, while in testing, it uses learned running statistics.
model.eval()

# printing and confirming the readiness of the device 
print("Model ready for inference.")

# Model Q4:

In [ ]:
preprocess = weights.transforms()
print(preprocess)

Below is a comment on the above snippet of code and an explanation of its output. 

Resize / Crop:
The image is first resized (usually the shorter side to 256), then center-cropped to 224x224.
This ensures every image has the exact same size expected by ResNet18.
It also helps focus on the image's main subject and removes unnecessary background edges.
Consistent sizing is critical because neural networks require fixed-size inputs.

ToTensor():
This converts a PIL image (values 0–255) into a PyTorch tensor.
It also rescales pixel values from [0, 255] → [0.0, 1.0].
This makes the data suitable for neural network computations (floating-point operations).

Normalization:
Normalization shifts and scales each channel (R, G, B) using:
   mean = [0.485, 0.456, 0.406]
   std  = [0.229, 0.224, 0.225]

This transforms the data so each channel has roughly zero mean and unit variance.
It helps the model learn more efficiently and keeps values in a stable range.

Why ImageNet mean/std specifically? 
Because the model was trained on ImageNet using these exact statistics.
Using the same normalization ensures the input data distribution matches what the model saw during training.
If you used mean=0.5 and std=0.5, the inputs would be shifted differently, which can hurt performance because the model would see "unfamiliar" data distributions.


# Running Inference

In [ ]:
import random
random.seed(42)

DATA_DIR = Path("/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test")
LABELS   = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

def load_sample_image(label):
    """Load a random image file from the given class folder."""
    class_dir = DATA_DIR / label
    img_path  = random.choice(list(class_dir.glob("*.jpg")))
    return Image.open(img_path).convert("RGB"), img_path.name


imagenet_classes = weights.meta["categories"]
print(f"Number of classes: {len(imagenet_classes)}")
print(f"First 5 labels: {imagenet_classes[:5]}")

In [ ]:
print(list(DATA_DIR.iterdir()))

In [ ]:
def get_top5_predictions(model, preprocess, image, device, class_labels):
    """
    Run inference on a PIL image and return the top-5 predictions.
    Returns a list of (class_name, probability) tuples.
    """
    # Step 1: Preprocess the image and add a batch dimension
    # (hint: use preprocess(), .unsqueeze(0), and .to(device))

    input_tensor = preprocess(image).unsqueeze(0).to(device)

    # Step 2: Run inference inside a torch.no_grad() block
    # (hint: call model() on your input tensor to get output of shape (1, 1000))

    with torch.no_grad():
        output = model(input_tensor)

    # Step 3: Convert raw scores (logits) to probabilities
    # (hint: use torch.nn.functional.softmax on output[0])

    probs = torch.nn.functional.softmax(output[0], dim=0)
    
    # Step 4: Get the top 5 predictions using torch.topk
    # (hint: returns top_probs and top_indices)
    
    top_probs, top_indices = torch.topk(probs, 5)

    # Step 5: Build and return a list of (class_name, probability) tuples
    results = []
    for prob, idx in zip(top_probs, top_indices):
        class_name = class_labels[idx.item()]
        prob_value = prob.item()
        results.append((class_name, prob_value))
    return results

img, img_name = load_sample_image("mountain")
preds         = get_top5_predictions(model, preprocess, img, device, imagenet_classes)

print(f"\nTop-5 predictions for '{img_name}':")
for class_name, prob in preds:
    print(f"  {class_name:30s}  {prob:.4f}")



The top prediction makes sense because "alp" is related to a mountain scene.
Other top-5 labels like "valley" and "mountain tent" also fit the image.
The model uses ImageNet labels, so it may not say "mountain" exactly.

# Inference Q2:

In [ ]:
for label in LABELS:
    img, img_name = load_sample_image(label)
    preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)[:3]
    print(f"\n[{label}]  {img_name}")
    for class_name, prob in preds:
        print(f"  {class_name:30s}  {prob:.4f}")

The model seems most confident on the mountain image, with a top-1 probability of 0.5933.
It seems least confident on the street image, with a top-1 probability of 0.1299.
A possible pattern is that natural scenes like mountains and glaciers match ImageNet labels more clearly, while city/street scenes may contain many objects, making the model less certain.

# Inference Q3:

In [ ]:
img, _ = load_sample_image("forest")
input_tensor = preprocess(img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_tensor)

probs = torch.nn.functional.softmax(logits[0], dim=0)

print(f"Logit  range: min={logits.min():.2f}, max={logits.max():.2f}")
print(f"Prob   range: min={probs.min():.6f}, max={probs.max():.4f}")
print(f"Probs sum to: {probs.sum():.6f}")
print(f"Top prediction: {imagenet_classes[probs.argmax().item()]}  ({probs.max():.4f})")

Neural networks output logits because they are better for training, allowing a flexible range of values without restriction.
Softmax converts them into probabilities that are easier to interpret.
In production, we would use probabilities because values between 0 and 1 make it easier to filter low-confidence predictions.

# Inference Q4:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img)
axes[0].axis("off")

class_names = [name for name, prob in preds]
probs = [prob for name, prob in preds]

axes[1].barh(class_names, probs)

axes[1].invert_yaxis()

axes[1].set_title("Top-5 Predictions")
axes[1].set_xlabel("Probability")

plt.tight_layout()
plt.savefig("outputs/warmup_inference_viz.png")

For a dashboard, I would keep the visualization simple with probabilities and clear labels so non-technical users can easily understand the predictions.
I would use a threshold of about 0.3 because many predictions are below 0.5, and 0.3 keeps reasonable predictions while still filtering out low-confidence ones.